# H1: effect of a notification on the proximal outcome in its own domain

For each domain d (mood, steps, sleep), the causal excursion effect of sending a
domain-d message vs. no message on the domain-d outcome in the following window,
averaged over all decision points (S_t empty):

* **Rows:** decisions with no message or a domain-d message (other-domain messages dropped).
* **Treatment probability:** P(A = 1 | H_t) = p_send · p_d / (p_send · p_d + p_no_send), known from the design.
* **Estimator:** DR-WCLS. Nuisance outcome models are cross-fitted by participant with
  `MRTpostInfLASSO`; the pseudo-outcome uses the known randomization probability and a
  constant reference probability p̃ = mean(P(A = 1)). The marginal effect is a weighted
  regression of the pseudo-outcome on an intercept with sandwich SEs clustered by participant.
* **Check:** WCLS with a linear control model on the same rows.
* **Secondary:** any message vs. no message on mood.

No moderator selection is involved, so the LASSO and post-selection steps of the package are not used here.

In [ ]:
library(yaml)
library(here)
library(readr)
library(dplyr)
library(tibble)
library(geepack)

remotes::install_github("WHD-Lab/DR_WCLS_LASSO@2295196c5f364302b0f9b29a154fbc77037daff8", dependencies = FALSE, upgrade = "never")
library(MRTpostInfLASSO)

source("h1_functions.R")

DATA_PATH <- read_yaml(here("config.yml"))$data_path
OUT_DIR   <- file.path(DATA_PATH, "results", "H1")
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

SEED   <- 2024
FOLDS  <- 5
METHOD <- "RandomForestLean"  # folds run one at a time, outcome model only; or package methods:
                              # "RandomForest", "CVLASSO", "GradientBoosting" (all folds in parallel)
CORES  <- 4                   # Lean: threads for ranger; package methods: parallel workers (one data copy each)

data <- read_csv(file.path(DATA_PATH, "processed/data_final_2024.csv"), show_col_types = FALSE)

### Checks on the treatment and probability columns

In [ ]:
print(table(data$message_metric, useNA = "ifany"))
print(table(data$device_type, useNA = "ifany"))

# map domains to the values used in `message_metric`
METRIC_VALUES <- list(mood = "Mood", steps = "Step Count", sleep = "Hours of Sleep")
stopifnot(all(unlist(METRIC_VALUES) %in% data$message_metric))
stopifnot(all(c("p_send", "p_no_send", paste0("p_", names(METRIC_VALUES))) %in% names(data)))

# do the domain probabilities vary within participants? (messages drawn without replacement)
data %>%
  filter(has_decision) %>%
  group_by(participant_id) %>%
  summarise(across(c(p_mood, p_steps, p_sleep), sd), .groups = "drop") %>%
  summarise(across(-participant_id, ~ mean(.x > 1e-8, na.rm = TRUE)))

### Outcomes and history

Step and sleep totals are set to missing when the window has data for fewer than
75% of its hours or is outside 20–28 hours long. Steps are analysed as log(1 + steps).
History variables are median-imputed with missingness indicators inside `run_h1`.

In [ ]:
data <- derive_variables(data, min_cov = 0.75, span_range = c(20, 28))

OUTCOMES <- c(mood = "Mood_post", steps = "log_steps_post", sleep = "sleep_minutes_post")

# TODO: decide whether to include day of week
Ht_candidates <- c(
  "Mood_pre", "Mood_sd_pre", "log_steps_pre", "sleep_minutes_pre", "hr_avg_pre", "RHR_pre",
  "Mood_n_pre", "steps_n_pre", "sleep_minutes_n_pre",
  "t", #"dow",
  "Age", "Sex", "depr0", "PHQtot0", "GADtot0", "Neu0_published", "SLE0"
)
missing_cols <- setdiff(Ht_candidates, names(data))
if (length(missing_cols)) message("not in data, skipped: ", paste(missing_cols, collapse = ", "))
Ht <- intersect(Ht_candidates, names(data))

# small linear control model for the WCLS check
Ht_lin <- intersect(c("Mood_pre", "log_steps_pre", "sleep_minutes_pre", "t", "PHQtot0"), Ht)

# summary of missing outcomes by domain
sapply(OUTCOMES, function(y) mean(is.na(data[[y]][data$has_decision])))

### Primary analysis: pooled devices

In [ ]:
specs <- tibble(
  domain  = c("mood", "steps", "sleep", "any"),
  outcome = c(OUTCOMES[["mood"]], OUTCOMES[["steps"]], OUTCOMES[["sleep"]], OUTCOMES[["mood"]]),
  role    = c("primary", "primary", "primary", "secondary")
)

results <- list()
for (i in seq_len(nrow(specs))) {
  s <- specs[i, ]
  message("H1: ", s$domain, " -> ", s$outcome)
  r <- run_h1(data, s$domain, s$outcome, Ht, Ht_lin, METRIC_VALUES,
              device = "pooled", method = METHOD,
              fold = FOLDS, core_num = CORES, seed = SEED)
  results[[i]] <- r$results %>% mutate(role = s$role, .before = 1)
  saveRDS(r$pseudo, file.path(OUT_DIR, sprintf("pseudo_%s_%s_pooled.rds", s$domain, s$outcome)))
}
h1 <- bind_rows(results)
write_csv(h1, file.path(OUT_DIR, "h1_pooled.csv"))

h1 %>% select(role, domain, outcome, estimator, n_participants, n_decisions,
              estimate, se, lower, upper, p_value, std_effect)

### Sensitivity: device-specific analyses

Device is not randomized, so differences between devices mix measurement with who uses which device.

In [ ]:
RUN_DEVICES <- TRUE
DEVICES <- c("Fitbit", "Garmin", "Healthkit", "HealthkitPhone", "HealthkitWatch")

if (RUN_DEVICES) {
  dev_results <- list()
  for (dv in DEVICES) for (i in which(specs$role == "primary")) {
    s <- specs[i, ]
    r <- run_h1(data, s$domain, s$outcome, Ht, Ht_lin, METRIC_VALUES,
                device = dv, method = METHOD,
                fold = FOLDS, core_num = CORES, seed = SEED)
    dev_results[[length(dev_results) + 1]] <- r$results
  }
  h1_dev <- bind_rows(dev_results)
  write_csv(h1_dev, file.path(OUT_DIR, "h1_by_device.csv"))
  h1_dev %>% select(device, domain, estimator, n_participants, estimate, se, lower, upper, p_value)
}